# 5. Storing Embeddings in a Vector Store (Chroma)

**RAG Pipeline Series — Notebook 5**

In notebook 4 we embedded `rag.pdf`'s chapter-tagged chunks into a NumPy array and searched it with a manual cosine-similarity loop. That's fine for a few dozen chunks, but it doesn't scale, doesn't persist to disk, and doesn't support filtering by metadata. A **vector store** handles all three. This notebook introduces **Chroma** — a lightweight, embedded vector database that plugs straight into LangChain — and shows it reproduces notebook 4's results in a single line of code.

In this notebook we will:
1. Rebuild the same chapter-tagged chunks used in notebooks 2, 3, and 4.
2. Wrap the notebook 4 embedding model in LangChain's `Embeddings` interface.
3. Index the chunks into a **Chroma** vector store, with chapter metadata attached.
4. Search it and compare directly against notebook 4's manual NumPy/cosine-similarity results on the same keyword-style and paraphrased queries.
5. Wrap the store as a **retriever**, the interface later notebooks in this series build on.

## Setup

In [1]:
%pip install -q -U langchain langchain-community pypdf sentence-transformers langchain-huggingface langchain-chroma chromadb pandas

Note: you may need to restart the kernel to use updated packages.


## 1. Recap: loading and chunking `rag.pdf`

Same loading + chapter-aware chunking as notebooks 2, 3, and 4: load pages with `PyPDFLoader`, strip the repeated header/footer lines, split the full text into per-chapter spans, then run `RecursiveCharacterTextSplitter` *within* each chapter so no chunk crosses a chapter boundary. Each chunk keeps its `chapter_num` / `chapter_title` as metadata — which we'll hand to Chroma so it's stored right alongside the vector.

In [ ]:
# Only runs inside Colab. Opens a file picker; select rag.pdf.
# Safe to skip this cell if you're running locally and already have the file on disk.
from rag_utils import maybe_colab_upload

maybe_colab_upload()

In [1]:
from rag_utils import resolve_pdf_path, load_clean_text

# Prefer the Colab upload location; fall back to this repo's dataset/ folder
# when running locally (rag-notebooks/ and dataset/ are sibling folders).
PDF_PATH = resolve_pdf_path()
pages, full_text = load_clean_text(PDF_PATH)

print(f"Loaded {len(pages)} pages from {PDF_PATH}")

d:\youtube\TheAIGuy\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 58 pages from ..\dataset\rag.pdf


In [2]:
from rag_utils import DEFAULT_CHUNK_OVERLAP, DEFAULT_CHUNK_SIZE, chunk_chapters, split_into_chapters

CHUNK_SIZE, CHUNK_OVERLAP = DEFAULT_CHUNK_SIZE, DEFAULT_CHUNK_OVERLAP
chapters = split_into_chapters(full_text)
chunks = chunk_chapters(chapters, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print(f"{len(chapters)} chapters -> {len(chunks)} chapter-tagged chunks")
print(chunks[0].metadata)
print(chunks[0].page_content[:200])

15 chapters -> 181 chapter-tagged chunks
{'chapter_num': '01', 'chapter_title': 'Introduction to RAG'}
Retrieval-Augmented Generation (RAG) is one of the most transformative patterns in modern AI
engineering. It bridges the gap between the remarkable language capabilities of Large Language
Models (LLMs


## 2. Wrapping the embedding model for LangChain

Notebook 4 called `sentence-transformers/paraphrase-MiniLM-L3-v2` directly. `HuggingFaceEmbeddings` wraps the same model behind LangChain's standard `Embeddings` interface (`embed_documents`, `embed_query`) so any LangChain-compatible vector store — Chroma included — can call it without knowing anything about `sentence-transformers` itself.

In [3]:
from rag_utils import DEFAULT_EMBEDDING_MODEL, get_embedder

EMBEDDING_MODEL = DEFAULT_EMBEDDING_MODEL  # same tiny model as notebook 4: ~17M params, 384-dim
embeddings = get_embedder(EMBEDDING_MODEL)

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 76.88it/s]


## 3. Building the vector store

`Chroma.from_documents` embeds every chunk and stores the vectors together with its metadata — in memory here (no `persist_directory`), so every notebook run starts from a clean, reproducible index.

In [4]:
from rag_utils import build_chroma_store

vectorstore = build_chroma_store(chunks, embeddings=embeddings, collection_name="rag_pdf_chapters")
print(f"Indexed {vectorstore._collection.count()} chunks into Chroma")

Indexed 181 chunks into Chroma


## 4. Searching — one line instead of a manual loop

Compare this to notebook 4, where we manually encoded the query, computed cosine similarity against a NumPy array, and sorted the results ourselves. `similarity_search_with_score` does all of that internally. We reuse the exact same target chunks and queries from notebooks 3 and 4, so the comparison is apples-to-apples.

In [5]:
bm25_chunk_idx = next(i for i, d in enumerate(chunks) if "Okapi Best Match 25" in d.page_content)
hallucination_chunk_idx = next(i for i, d in enumerate(chunks) if "dynamic, external knowledge source" in d.page_content)

keyword_query = "Okapi Best Match 25 term frequency saturation document length normalization"
paraphrase_query = "How can giving a language model outside documents stop it from making things up?"

print("Keyword-style query:", keyword_query)
print("Target chunk metadata:", chunks[bm25_chunk_idx].metadata)
for doc, score in vectorstore.similarity_search_with_score(keyword_query, k=5):
    print(f"  score={score:.4f}  {doc.metadata}")

Keyword-style query: Okapi Best Match 25 term frequency saturation document length normalization
Target chunk metadata: {'chapter_num': '02', 'chapter_title': 'Evolution of Retrieval'}
  score=1.0316  {'chapter_num': '02', 'chapter_title': 'Evolution of Retrieval'}
  score=1.2892  {'chapter_num': '05', 'chapter_title': 'Vector Databases & Indexing'}
  score=1.2893  {'chapter_num': '08', 'chapter_title': 'Re-ranking'}
  score=1.3460  {'chapter_num': '06', 'chapter_title': 'Retrieval Techniques'}
  score=1.3495  {'chapter_num': '04', 'chapter_title': 'Embeddings'}


In [6]:
print("Paraphrased query:", paraphrase_query)
print("Target chunk metadata:", chunks[hallucination_chunk_idx].metadata)
for doc, score in vectorstore.similarity_search_with_score(paraphrase_query, k=5):
    print(f"  score={score:.4f}  {doc.metadata}")
print()
print("Compare these rankings to notebook 4's manual cosine-similarity search on the same queries — same chunks should come out on top, since it's the same model over the same chunks.")

Paraphrased query: How can giving a language model outside documents stop it from making things up?
Target chunk metadata: {'chapter_num': '01', 'chapter_title': 'Introduction to RAG'}
  score=1.1155  {'chapter_title': 'Data Ingestion', 'chapter_num': '03'}
  score=1.1199  {'chapter_title': 'Introduction to RAG', 'chapter_num': '01'}
  score=1.1977  {'chapter_title': 'Augmentation', 'chapter_num': '09'}
  score=1.2045  {'chapter_num': '03', 'chapter_title': 'Data Ingestion'}
  score=1.2159  {'chapter_title': 'Introduction to RAG', 'chapter_num': '01'}

Compare these rankings to notebook 4's manual cosine-similarity search on the same queries — same chunks should come out on top, since it's the same model over the same chunks.


Note the score convention: Chroma's default distance metric here is **squared L2 (Euclidean) distance**, not cosine similarity, so *lower* scores mean *more* similar — the opposite direction from notebook 4's cosine scores. Always check which convention a vector store uses before comparing numbers across tools.

## 5. Rank comparison: where does the correct chunk land?

Same rank-of-correct-chunk comparison as notebooks 3 and 4, now with Chroma's results as the source of ranks.

In [7]:
import pandas as pd

def rank_of(target_idx, query, k=10):
    results = vectorstore.similarity_search_with_score(query, k=k)
    for rank, (doc, score) in enumerate(results, start=1):
        if doc.page_content == chunks[target_idx].page_content:
            return rank
    return f"> {k}"

rows = [
    {"query": "Keyword-style", "chroma_rank": rank_of(bm25_chunk_idx, keyword_query)},
    {"query": "Paraphrased", "chroma_rank": rank_of(hallucination_chunk_idx, paraphrase_query)},
]
pd.DataFrame(rows)

,query,chroma_rank
0,Keyword-style,1
1,Paraphrased,> 10


## 6. Retriever interface

LangChain chains expect a `Retriever`, not a raw vector store. `.as_retriever()` wraps the vector store so it can be dropped straight into an LCEL chain — the interface we'll build on in later notebooks in this series.

In [8]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
retrieved = retriever.invoke(paraphrase_query)
[d.metadata for d in retrieved]

[{'chapter_title': 'Data Ingestion', 'chapter_num': '03'},
 {'chapter_num': '01', 'chapter_title': 'Introduction to RAG'},
 {'chapter_title': 'Augmentation', 'chapter_num': '09'}]

## Takeaways

- A vector store bundles the embedding model, the storage, and the similarity search into one object — no manual NumPy loop required, and metadata (`chapter_num`, `chapter_title`) travels with each vector automatically.
- Chroma's default score is a *distance* (lower = more similar), the reverse of cosine *similarity* (higher = more similar) used in notebook 4 — always check the convention before comparing numbers across tools.
- `.as_retriever()` is how a vector store plugs into LangChain chains in later notebooks.

**Next up (notebook 6):** using the `chapter_num` / `chapter_title` metadata attached to every chunk to **filter** searches — e.g., "only search within one chapter."